# Eshmun Supervised Fine-Tuning — TRL

SFT on `khairi/eshmun-instructions` starting from `khairi/Eshmun-0.3B-CPT-LoRA`.

**Strategy:**
1. Load base model and merge the CPT-LoRA adapter
2. Inject fresh SFT-LoRA adapters
3. `SFTTrainer` handles everything: chat-template application, tokenization, sequence packing, and padding
   — the `formatting_func` is the only custom piece, converting `(Input, Output)` pairs to a
   chat-templated string on the fly

**Pipeline:**
1. Install dependencies
2. Login to HuggingFace Hub
3. Imports
4. Configuration
5. Load model — merge CPT-LoRA, inject SFT-LoRA
6. Dataset
7. Formatting function (chat template on the fly)
8. Training with `SFTTrainer`
9. Save and push to Hub

## 1. Install dependencies

In [ ]:
!pip install -q git+https://github.com/abidikhairi/eshmun.git
!pip install -q datasets transformers accelerate peft trl

## 2. Login to HuggingFace Hub

In [ ]:
from huggingface_hub import login as hf_login

hf_login()  # paste your HF write-access token when prompted

## 3. Imports

In [ ]:
import torch
from datasets import load_dataset
from peft import LoraConfig, PeftModel, TaskType, get_peft_model
from transformers import AutoModelForCausalLM, AutoTokenizer
from trl import SFTConfig, SFTTrainer

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 4. Configuration

In [ ]:
BASE_MODEL_ID  = "khairi/Eshmun-0.3B-Base"       # base weights (pre-CPT)
CPT_LORA_ID    = "khairi/Eshmun-0.3B-CPT-LoRA"   # CPT adapter to merge in
DATASET_ID     = "khairi/eshmun-instructions"
HF_REPO_ID     = "khairi/Eshmun-0.3B-SFT"        # SFT adapter pushed here
OUTPUT_DIR     = "/tmp/eshmun-0.3b-sft-trl"

MAX_SEQ_LEN = 512

# LoRA hyperparameters
LORA_R       = 16
LORA_ALPHA   = 32
LORA_DROPOUT = 0.05

# Training hyperparameters
PER_DEVICE_TRAIN_BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 8       # effective batch = 32
NUM_TRAIN_EPOCHS            = 3
LEARNING_RATE               = 2e-4
WARMUP_STEPS                = 100
LOGGING_STEPS               = 50
SAVE_STEPS                  = 500

## 5. Model & tokenizer

1. Load `BASE_MODEL_ID`
2. Attach the CPT-LoRA adapter and merge it into the base weights
3. Wrap with fresh SFT-LoRA adapters

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(CPT_LORA_ID, trust_remote_code=True)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float32,
)

# Merge CPT-LoRA into base weights, then discard adapter scaffolding
model = PeftModel.from_pretrained(base_model, CPT_LORA_ID)
model = model.merge_and_unload()

n_params = sum(p.numel() for p in model.parameters())
print(f"Merged model: {n_params / 1e6:.1f}M parameters")

## 6. SFT-LoRA setup

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "out_proj", "fc1", "fc2"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 7. Dataset

In [ ]:
dataset = load_dataset(DATASET_ID, split="train")
dataset = dataset.select_columns(["Input", "Output"])

print(dataset)
print(dataset[0])

## 8. Formatting function

`SFTTrainer` calls this once per example to produce the text that gets tokenized and packed.
The chat template is applied here — no pre-processing step needed.

In [ ]:
def format_example(example: dict) -> str:
    messages = [
        {"role": "user",      "content": example["Input"]},
        {"role": "assistant", "content": example["Output"]},
    ]
    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )


# Sanity check
print(format_example(dataset[0]))

## 9. Training

`SFTTrainer` with `packing=True` concatenates examples into `max_seq_length` chunks,
maximising GPU utilisation. Tokenization and padding are handled internally.

In [ ]:
sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    # SFT-specific
    max_seq_length=MAX_SEQ_LEN,
    packing=True,
    # General training
    num_train_epochs=NUM_TRAIN_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    gradient_checkpointing=True,
    learning_rate=LEARNING_RATE,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    max_grad_norm=1.0,
    lr_scheduler_type="cosine",
    fp16=False,
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    push_to_hub=True,
    hub_model_id=HF_REPO_ID,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=dataset,
    processing_class=tokenizer,
    formatting_func=format_example,
)

trainer.train()

## 10. Save and push to Hub

`model.save_pretrained` saves only the **SFT adapter weights**. Load it back with:
```python
from peft import PeftModel
base   = AutoModelForCausalLM.from_pretrained("khairi/Eshmun-0.3B-Base", trust_remote_code=True)
cpt    = PeftModel.from_pretrained(base, "khairi/Eshmun-0.3B-CPT-LoRA")
merged = cpt.merge_and_unload()
model  = PeftModel.from_pretrained(merged, "khairi/Eshmun-0.3B-SFT")
```

In [ ]:
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"SFT adapter saved to {OUTPUT_DIR}")

trainer.push_to_hub(commit_message="sft trl lora checkpoint")
print(f"SFT adapter pushed to https://huggingface.co/{HF_REPO_ID}")